In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter

# Paths
TRAIN_PATH = '/kaggle/input/competitions/datasprint/train/train'
TEST_PATH = '/kaggle/input/competitions/datasprint/test/test'

# Classes define karo
CLASSES = ['glioma_tumor', 'meningioma_tumor', 'pituitary_tumor', 'no_tumor']
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}

print("Label Mapping:")
for k, v in LABEL_MAP.items():
    print(f"  {k} → {v}")

# Train images count
train_files = os.listdir(TRAIN_PATH)
test_files = os.listdir(TEST_PATH)

print(f"\nTotal Train Images: {len(train_files)}")
print(f"Total Test Images:  {len(test_files)}")

# Class Distribution

In [ ]:
# extract label from filename
def get_label(filename):
    for cls in CLASSES:
        if cls in filename:
            return cls
    return 'unknown'

# Count karo
train_labels = [get_label(f) for f in train_files]
label_counts = Counter(train_labels)

print("Train Class Distribution:")
for cls, count in label_counts.items():
    pct = count / len(train_files) * 100
    print(f"  {cls}: {count} images ({pct:.1f}%)")

# Plot karo
plt.figure(figsize=(8, 4))
plt.bar(label_counts.keys(), label_counts.values(), color=['red','blue','green','gray'])
plt.title('Class Distribution in Training Data')
plt.xticks(rotation=15)
plt.ylabel('Count')
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from PIL import Image

# ─── Config ───────────────────────────────────────
IMG_SIZE   = 128
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 0.001
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASSES  = ['glioma_tumor','meningioma_tumor','pituitary_tumor','no_tumor']
LABEL_MAP = {c: i for i, c in enumerate(CLASSES)}

print(f"Device: {DEVICE}")

# Dataset Class

In [ ]:
class BrainMRIDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Image load 
        img = Image.open(self.file_paths[idx]).convert('L')  # Grayscale

        if self.transform:
            img = self.transform(img)

        label = self.labels[idx]
        return img, label

# Data Loading + Augmentation

In [ ]:
import os

# Collect all files and labels
all_paths, all_labels = [], []

for fname in os.listdir(TRAIN_PATH):
    label = get_label(fname)
    if label != 'unknown':
        all_paths.append(os.path.join(TRAIN_PATH, fname))
        all_labels.append(LABEL_MAP[label])

# Train/Validation split — 80/20
X_train, X_val, y_train, y_val = train_test_split(
    all_paths, all_labels,
    test_size=0.2,
    random_state=42,
    stratify=all_labels  # maintain class balance 
)

print(f"Train: {len(X_train)} | Val: {len(X_val)}")

# Augmentation — for training 
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# for Validation — only resize and normalize
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Dataset objects
train_dataset = BrainMRIDataset(X_train, y_train, train_transform)
val_dataset   = BrainMRIDataset(X_val,   y_val,   val_transform)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders ready! ✅")

# CNN Model — Cell 7

In [ ]:
class BrainTumorCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),          # 128→64
            nn.Dropout(0.25),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),          # 64→32
            nn.Dropout(0.25),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),          # 32→16
            nn.Dropout(0.4),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 4)            # 4 classes!
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = BrainTumorCNN().to(DEVICE)
print(model)
print(f"\nTotal Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights — no_tumor ko zyada attention denge
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print("Class Weights:", class_weights)

# Loss aur Optimizer
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

# Training Loop

In [ ]:
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # ── Training ──────────────────────────────
    model.train()
    train_loss, train_correct = 0, 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()

    # ── Validation ────────────────────────────
    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images  = images.to(DEVICE)
            labels  = labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)

            val_loss    += loss.item()
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()

    # ── Metrics ───────────────────────────────
    train_acc = train_correct / len(X_train) * 100
    val_acc   = val_correct   / len(X_val)   * 100

    scheduler.step(val_acc)

    # Best model save karo
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Acc: {train_acc:.1f}% | "
          f"Val Acc: {val_acc:.1f}% | "
          f"Best: {best_val_acc:.1f}%")

print(f"\n✅ Training Complete! Best Val Accuracy: {best_val_acc:.1f}%")

# Evaluation

In [ ]:
from sklearn.metrics import (confusion_matrix, accuracy_score,
                              precision_score, f1_score,
                              classification_report)
import seaborn as sns

# Best model load
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds, all_labels_list = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels_list.extend(labels.numpy())

# Metrics
print("✅ Accuracy :", accuracy_score(all_labels_list, all_preds))
print("✅ Precision:", precision_score(all_labels_list, all_preds, average='weighted'))
print("✅ F1 Score :", f1_score(all_labels_list, all_preds, average='weighted'))
print("\n📊 Classification Report:")
print(classification_report(all_labels_list, all_preds, target_names=CLASSES))

# Confusion Matrix
cm = confusion_matrix(all_labels_list, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Test images load 
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

test_files_list = sorted(os.listdir(TEST_PATH))
image_ids, predictions = [], []

model.eval()
with torch.no_grad():
    for fname in test_files_list:
        img_path = os.path.join(TEST_PATH, fname)
        img = Image.open(img_path).convert('L')
        img = test_transform(img).unsqueeze(0).to(DEVICE)

        output = model(img)
        pred_idx = output.argmax(dim=1).item()
        pred_label = CLASSES[pred_idx]

        # image_id = filename without extension
        image_id = fname
        image_ids.append(image_id)
        predictions.append(pred_label)

# Submission CSV
submission = pd.DataFrame({
    'image_id': image_ids,
    'label': predictions
})

submission.to_csv('submission.csv', index=False)
print(f"✅ Submission file ready! Shape: {submission.shape}")
print(submission.head(10))

In [ ]:
submission2 = pd.DataFrame({
    'image_id': image_ids,
    'label': predictions
})

# Step 1: clean image_id 
def clean_image_id(fname):
    for cls in CLASSES:
        fname = fname.replace(f'_{cls}', '')
    fname = fname.replace('.png', '')
    return fname

# Step 2: make label's to binary
def convert_to_binary(label):
    if label == 'no_tumor':
        return 'No Tumor'
    else:
        return 'Tumor'

submission2['image_id'] = submission2['image_id'].apply(clean_image_id)
submission2['label']    = submission2['label'].apply(convert_to_binary)

submission2.to_csv('submission.csv', index=False)
print(submission2.head(10))
print(f"\nLabel Distribution:")
print(submission2['label'].value_counts())